## Notebook 01: ESCO Import & Baseline-Tables

Notebooks 01–04 form the foundation of the entire project.  ESCO serves here (alongside KldB) as the official, standardized reference framework for occupational and competency profiles.  All subsequent target profiles, as well as the profile extension, are based on these structured ESCO master tables:
- Download ESCO v1.2.0 (Classification, CSV, en), November 14, 2025
- Prepare occupations (`occupations_en`), skills (`skills_en`), and relationships (`occupationSkillRelations_en`)
- Create a base schema for: Occupation-Skill Profiles (OccupationSkills), KldB -> ESCO mapping, profile extension

(Sources: ESCO, https://esco.ec.europa.eu/en; https://esco.ec.europa.eu/en/classification; https://esco.ec.europa.eu/en/structure-esco-downloadable-datasets; https://esco.ec.europa.eu/en/about-esco/data-science-and-esco/esco-skill-occupation-matrix-tables-linking-occupation-and-skill-groups; 14.11.2025)

In [1]:
# imports
import sys
import pandas as pd
from pathlib import Path

# Kernel Check
print(sys.version)
print(pd.__version__)

3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
2.3.3


In [2]:
# Project Root
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "esco"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

print("Project root:", PROJECT_ROOT)
print("ESCO raw path:", DATA_RAW)
print("Interim path:", DATA_INTERIM)

DATA_INTERIM.mkdir(parents=True, exist_ok=True)

Project root: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
ESCO raw path: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw\esco
Interim path: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim


## 1. Load and view raw data

In [3]:
list(DATA_RAW.glob("*.csv")) # ESCO files

[WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/esco/broaderRelationsOccPillar_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/esco/broaderRelationsSkillPillar_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/esco/conceptSchemes_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/esco/digCompSkillsCollection_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/esco/digitalSkillsCollection_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/esco/greenSkillsCollection_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/esco/ISCOGroups_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule R

In [4]:
# Helper function, further customized, separator
def read_esco_csv(path: Path) -> pd.DataFrame: # Reads ESCO-CSV files and cycles through separators (; , Tab) with engine='python'
    print(f"\nLoading {path.name} ...")

    for sep in [";", ",", "\t"]:
        try:
            print(f"  Trying sep='{sep}' ...")
            df = pd.read_csv(path, sep=sep, dtype=str, engine="python")
            print(f"  -> success with sep='{sep}': {df.shape[0]} rows x {df.shape[1]} columns")
            return df
        except Exception as e:
            print(f"  -> failed with sep='{sep}': {e}")

    raise RuntimeError(f"Konnte Datei {path} mit den getesteten Separators nicht einlesen.")

In [5]:
#Raw data
occ_path = DATA_RAW / "occupations_en.csv"
skill_path = DATA_RAW / "skills_en.csv"
rel_path  = DATA_RAW / "occupationSkillRelations_en.csv"

occupations_raw = read_esco_csv(occ_path)
skills_raw      = read_esco_csv(skill_path)
relations_raw   = read_esco_csv(rel_path)


Loading occupations_en.csv ...
  Trying sep=';' ...
  -> failed with sep=';': ';' expected after '"'
  Trying sep=',' ...
  -> success with sep=',': 3039 rows x 14 columns

Loading skills_en.csv ...
  Trying sep=';' ...
  -> failed with sep=';': ';' expected after '"'
  Trying sep=',' ...
  -> success with sep=',': 13939 rows x 13 columns

Loading occupationSkillRelations_en.csv ...
  Trying sep=';' ...
  -> success with sep=';': 129004 rows x 1 columns


In [6]:
print(occupations_raw.columns.tolist()) # Columns in occupations

['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code']


In [7]:
# Columns in the three main files
print("Occupations columns:")
print(occupations_raw.columns.tolist())

print("\nSkills columns:")
print(skills_raw.columns.tolist())

print("\nRelations columns:")
print(relations_raw.columns.tolist())

Occupations columns:
['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code']

Skills columns:
['conceptType', 'conceptUri', 'skillType', 'reuseLevel', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'scopeNote', 'definition', 'inScheme', 'description']

Relations columns:
['occupationUri,relationType,skillType,skillUri']


In [8]:
# Sample output
display(occupations_raw.head(3))
display(skills_raw.head(3))
display(relations_raw.head(3))

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,technical and operations director\nhead of tec...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Technical directors realise the artistic visio...,2654.1.7
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,metal drawing machine technician\nmetal drawin...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,inspector of precision instruments\nprecision ...,NaN,released,2024-01-25T15:00:12.188Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Precision device inspectors make sure precisio...,7543.10.3


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage staff of music\ncoordinate duties of mu...,NaN,released,2023-11-30T15:53:37.136Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,oversee prison procedures\nmanage correctional...,NaN,released,2023-11-30T15:04:00.689Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Supervise the operations of a correctional fac...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,apply non-oppressive practices\napply an anti-...,NaN,released,2023-11-28T10:45:53.54Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c..."


,"occupationUri,relationType,skillType,skillUri"
0,http://data.europa.eu/esco/occupation/00030d09...
1,http://data.europa.eu/esco/occupation/00030d09...
2,http://data.europa.eu/esco/occupation/00030d09...


## 2. Processing the Occupations Table

DataFrame, extracting the most important fields:
- `conceptUri` -> `occupation_uri` (primary key)
- `code` -> `occupation_code` (ESCO occupation code)
- `preferredLabel` (main label)
- `altLabels` (synonyms, variants)
- `description` (short description)

In [9]:
required_occ_cols = ["conceptUri", "preferredLabel"]
for col in required_occ_cols:
    if col not in occupations_raw.columns:
        raise ValueError(f"Erwartete Spalte '{col}' in occupations_raw nicht gefunden.")

esco_occupations = pd.DataFrame({
    "occupation_uri": occupations_raw["conceptUri"].astype(str).str.strip(),
    "occupation_code": occupations_raw.get("code", pd.NA),
    "isco_group": occupations_raw.get("iscoGroup", pd.NA),  # ISCO group for later mapping
    "pref_label_en": occupations_raw["preferredLabel"].astype(str).str.strip(),
    "alt_labels_en": occupations_raw.get("altLabels", pd.NA),
    "description_en": occupations_raw.get("description", pd.NA),
})

# NaN -> empty strings for text columns
text_cols = ["pref_label_en", "alt_labels_en", "description_en"]
for col in text_cols:
    if col in esco_occupations.columns:
        esco_occupations[col] = esco_occupations[col].fillna("").astype(str).str.strip()

print(esco_occupations.shape)
esco_occupations.head(5)

(3039, 6)


,occupation_uri,occupation_code,isco_group,pref_label_en,alt_labels_en,description_en
0,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,technical and operations director\nhead of tec...,Technical directors realise the artistic visio...
1,http://data.europa.eu/esco/occupation/000e93a3...,8121.4,8121,metal drawing machine operator,metal drawing machine technician\nmetal drawin...,Metal drawing machine operators set up and ope...
2,http://data.europa.eu/esco/occupation/0019b951...,7543.10.3,7543,precision device inspector,inspector of precision instruments\nprecision ...,Precision device inspectors make sure precisio...
3,http://data.europa.eu/esco/occupation/0022f466...,3155.1,3155,air traffic safety technician,air traffic safety electronics hardware specia...,Air traffic safety technicians provide technic...
4,http://data.europa.eu/esco/occupation/002da35b...,2431.9,2431,hospitality revenue manager,hospitality revenues manager\nyield manager\nh...,Hospitality revenue managers maximise revenue ...


## 3. Skills Table Format

Extracted based on the key fields of the ESCO skills

In [10]:
required_skill_cols = ["conceptUri", "preferredLabel"]
for col in required_skill_cols:
    if col not in skills_raw.columns:
        raise ValueError(f"Erwartete Spalte '{col}' in skills_raw nicht gefunden.")

esco_skills = pd.DataFrame({
    "skill_uri": skills_raw["conceptUri"].astype(str).str.strip(),
    "pref_label_en": skills_raw["preferredLabel"].astype(str).str.strip(),
    "alt_labels_en": skills_raw.get("altLabels", pd.NA),
    "description_en": skills_raw.get("description", pd.NA),
    "reuse_level": skills_raw.get("reuseLevel", pd.NA),
    "skill_type": skills_raw.get("skillType", pd.NA),
})

text_cols = ["pref_label_en", "alt_labels_en", "description_en"]
for col in text_cols:
    if col in esco_skills.columns:
        esco_skills[col] = esco_skills[col].fillna("").astype(str).str.strip()

print(esco_skills.shape)
esco_skills.head(5)

(13939, 6)


,skill_uri,pref_label_en,alt_labels_en,description_en,reuse_level,skill_type
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,manage staff of music\ncoordinate duties of mu...,Assign and manage staff tasks in areas such as...,sector-specific,skill/competence
1,http://data.europa.eu/esco/skill/00064735-8fad...,supervise correctional procedures,oversee prison procedures\nmanage correctional...,Supervise the operations of a correctional fac...,occupation-specific,skill/competence
2,http://data.europa.eu/esco/skill/000709ed-2be5...,apply anti-oppressive practices,apply non-oppressive practices\napply an anti-...,"Identify oppression in societies, economies, c...",sector-specific,skill/competence
3,http://data.europa.eu/esco/skill/0007bdc2-dd15...,control compliance of railway vehicles regulat...,monitoring of compliance with railway vehicles...,"Inspect rolling stock, components and systems ...",sector-specific,skill/competence
4,http://data.europa.eu/esco/skill/00090cc1-1f27...,identify available services,establish available services\ndetermine rehabi...,Identify the different services available for ...,cross-sector,skill/competence


`skill_type` distinguishes between knowledge elements and skill/competence elements.  
`reuse_level` indicates whether a skill is sector-specific or cross-sector. 

This metadata enables a more nuanced analysis and interpretation of the profiles.

## 4. Processing Occupation-Skill Relationships

Table links ESCO occupations with associated skills; includes the `relationType` attribute (e.g., `essential` vs. `optional`)

In [11]:
# Load the relations file explicitly using a comma as the separator
rel_path = DATA_RAW / "occupationSkillRelations_en.csv"

relations_raw = pd.read_csv(rel_path, sep=",", dtype=str)

print("Relations columns:")
print(relations_raw.columns.tolist())
display(relations_raw.head(3))

Relations columns:
['occupationUri', 'relationType', 'skillType', 'skillUri']


,occupationUri,relationType,skillType,skillUri
0,http://data.europa.eu/esco/occupation/00030d09...,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...
1,http://data.europa.eu/esco/occupation/00030d09...,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...
2,http://data.europa.eu/esco/occupation/00030d09...,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...


In [12]:
required_rel_cols = ["occupationUri", "skillUri"]
for col in required_rel_cols:
    if col not in relations_raw.columns:
        raise ValueError(f"Spalte '{col}' in relations_raw nicht gefunden")

occupation_skill_relations = pd.DataFrame({
    "occupation_uri": relations_raw["occupationUri"].astype(str).str.strip(),
    "skill_uri": relations_raw["skillUri"].astype(str).str.strip(),
    "relation_type": relations_raw.get("relationType", "").astype(str).str.lower().str.strip(),
})

print(occupation_skill_relations["relation_type"].value_counts(dropna=False))
occupation_skill_relations.head(5)

relation_type
essential    67622
optional     61382
Name: count, dtype: int64


,occupation_uri,skill_uri,relation_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/271a36a0-bc7a...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/47ed1d37-971b...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/591dd514-735b...,essential


## 5. Consistency Check

Check whether all `occupation_uri` and `skill_uri` from the relations are also present in the corresponding master tables

In [13]:
occ_in_rel = set(occupation_skill_relations["occupation_uri"])
occ_all = set(esco_occupations["occupation_uri"])
missing_occ = occ_in_rel - occ_all

skill_in_rel = set(occupation_skill_relations["skill_uri"])
skill_all = set(esco_skills["skill_uri"])
missing_skills = skill_in_rel - skill_all

print("URIs in relations, die nicht in occupations vorkommen:", len(missing_occ))
print("URIs in relations, die nicht in skills vorkommen:", len(missing_skills))

URIs in relations, die nicht in occupations vorkommen: 0
URIs in relations, die nicht in skills vorkommen: 0


As expected = 0, i.e., no missing URIs

This step ensures that there are no orphaned references, i.e., every `occupation_uri` or `skill_uri` referenced in the relations must exist in the respective master tables.
 

## 6. Quality Check

In [14]:
# Number of unique IDs
print("Unique occupation_uri:", esco_occupations["occupation_uri"].nunique())
print("Unique skill_uri:", esco_skills["skill_uri"].nunique())
print("Unique relations (pairs):", occupation_skill_relations[["occupation_uri","skill_uri"]].drop_duplicates().shape[0])

# Check empty texts
for col in ["pref_label_en", "description_en"]:
    empty = (esco_occupations[col] == "").sum()
    print(f"Leere Texte in {col}: {empty}")

# Relationships: essential vs. optional
print("\nRelation types:")
print(occupation_skill_relations["relation_type"].value_counts())

# Distribution of Skill Types
print("\nSkill types:")
print(esco_skills["skill_type"].value_counts())

Unique occupation_uri: 3039
Unique skill_uri: 13939
Unique relations (pairs): 128987
Leere Texte in pref_label_en: 0
Leere Texte in description_en: 0

Relation types:
relation_type
essential    67622
optional     61382
Name: count, dtype: int64

Skill types:
skill_type
skill/competence    10715
knowledge            3219
Name: count, dtype: int64


- Number of unique occupation and skill URIs: 3,039 occupations and 13,939 different skills
- Categories (e.g., essential/optional; skillType)

## 7. Convert altLabels to lists

For later text-matching processes, the `altLabels` can be converted into lists of synonyms (separator: |)

In [15]:
def split_alt_labels(series: pd.Series) -> pd.Series: # old label strings, separated by ‘|’, ignores empty entries
    return (
        series.fillna("")
        .astype(str)
        .apply(lambda x: [s.strip() for s in x.split("|") if s.strip()] if x else [])
    )

if "alt_labels_en" in esco_occupations.columns:
    esco_occupations["alt_labels_en"] = split_alt_labels(esco_occupations["alt_labels_en"])

if "alt_labels_en" in esco_skills.columns:
    esco_skills["alt_labels_en"] = split_alt_labels(esco_skills["alt_labels_en"])

esco_occupations.head(3)

,occupation_uri,occupation_code,isco_group,pref_label_en,alt_labels_en,description_en
0,http://data.europa.eu/esco/occupation/00030d09...,2654.1.7,2654,technical director,[technical and operations director\nhead of te...,Technical directors realise the artistic visio...
1,http://data.europa.eu/esco/occupation/000e93a3...,8121.4,8121,metal drawing machine operator,[metal drawing machine technician\nmetal drawi...,Metal drawing machine operators set up and ope...
2,http://data.europa.eu/esco/occupation/0019b951...,7543.10.3,7543,precision device inspector,[inspector of precision instruments\nprecision...,Precision device inspectors make sure precisio...


## 8. Storage of ESCO Base Tables

The cleaned tables are stored as Parquet files in `data/interim/` and serve as the basis: The Parquet format is used for storage because it is compressed and schema-based, which allows even large tables to be loaded and processed efficiently.

In [16]:
esco_occupations.to_parquet(DATA_INTERIM / "esco_occupations.parquet", index=False)
esco_skills.to_parquet(DATA_INTERIM / "esco_skills.parquet", index=False)
occupation_skill_relations.to_parquet(DATA_INTERIM / "occupation_skill_relations.parquet", index=False)

print("Gespeichert in:", DATA_INTERIM)

Gespeichert in: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim


(Note: In rare cases, after a period of inactivity or changes to the environment, sporadic kernel resets may occur in the saved notebook session during export. The code execution itself is not affected by this. A complete re-run is possible at any time; results remain unchanged and reproducible when re-run.)

# Summary Notebook 01

- ESCO v1.2.0 (Classification, CSV, en) has been transferred to three main tables:
  - `esco_occupations` (occupations)
  - `esco_skills` (skills)
  - `occupation_skill_relations` (occupation-skill relationships, including `relation_type`).
- Tables were cleaned up (trimming, optional synonym lists from `altLabels`) and standardized into a uniform schema; light data cleaning was performed (whitespace removed (.str.strip()), missing strings set to “” (text fields), columns standardized (e.g., occupation_uri), duplicate synonyms from altLabels converted to lists)
- All 3 tables saved as Parquet files in `data/interim/` and they form the basis for the next steps.